In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

project = Path.cwd().parent

activity = pd.read_csv(project / "data" / "processed" / "monthly_activity_clean.csv")
profile = pd.read_csv(project / "data" / "processed" / "customer_profile_clean.csv")
churn = pd.read_csv(project / "data" / "processed" / "churn_label_clean.csv")

activity["MONTH"] = pd.to_datetime(activity["MONTH"])
profile["SIGNUP_DATE"] = pd.to_datetime(profile["SIGNUP_DATE"])
churn["OBSERVATION_END_DATE"] = pd.to_datetime(churn["OBSERVATION_END_DATE"])
churn["LAST_PURCHASE_DATE"] = pd.to_datetime(churn["LAST_PURCHASE_DATE"])

duplicates = activity[
    activity.duplicated(["CUSTOMER_ID", "MONTH"], keep=False)
].sort_values(["CUSTOMER_ID", "MONTH"])

negative_spend_rows = activity[activity["TOTAL_SPEND"] < 0].copy()

profile_missing = profile.isna().sum()
activity_missing = activity.isna().sum()
churn_missing = churn.isna().sum()

churn_summary = (
    churn["CHURNED"]
    .value_counts()
    .rename_axis("CHURNED")
    .reset_index(name="CUSTOMERS")
)

churn_summary["PERCENT"] = (
    churn_summary["CUSTOMERS"] /
    churn_summary["CUSTOMERS"].sum() * 100
).round(2)

tier_churn = (
    profile[["CUSTOMER_ID", "MEMBERSHIP_TIER"]]
    .merge(churn[["CUSTOMER_ID", "CHURNED"]], on="CUSTOMER_ID")
    .groupby("MEMBERSHIP_TIER")
    .agg(
        CUSTOMERS=("CUSTOMER_ID", "nunique"),
        CHURN_RATE=("CHURNED", "mean")
    )
    .reset_index()
)

tier_churn["CHURN_RATE"] = (tier_churn["CHURN_RATE"] * 100).round(2)

city_churn = (
    profile[["CUSTOMER_ID", "CITY"]]
    .merge(churn[["CUSTOMER_ID", "CHURNED"]], on="CUSTOMER_ID")
    .groupby("CITY")
    .agg(
        CUSTOMERS=("CUSTOMER_ID", "nunique"),
        CHURN_RATE=("CHURNED", "mean")
    )
    .reset_index()
)

city_churn["CHURN_RATE"] = (city_churn["CHURN_RATE"] * 100).round(2)

monthly_activity = (
    activity.groupby("MONTH")
    .agg(
        CUSTOMERS=("CUSTOMER_ID", "nunique"),
        TRANSACTIONS=("TRANSACTIONS", "sum"),
        TOTAL_SPEND=("TOTAL_SPEND", "sum"),
        AVG_BASKET_VALUE=("AVG_BASKET_VALUE", "mean"),
        APP_SESSIONS=("APP_SESSIONS", "sum"),
        EMAILS_OPENED=("EMAILS_OPENED", "sum"),
        COUPONS_REDEEMED=("COUPONS_REDEEMED", "sum"),
        SUPPORT_TICKETS=("SUPPORT_TICKETS", "sum")
    )
    .reset_index()
)

missing_values = pd.DataFrame({
    "Profile Missing": profile_missing,
    "Activity Missing": activity_missing,
    "Churn Missing": churn_missing
}).fillna(0).astype(int)

display(churn_summary)
display(missing_values)
display(tier_churn)
display(city_churn.sort_values("CHURN_RATE", ascending=False))
display(duplicates)
display(negative_spend_rows)
display(monthly_activity)

,CHURNED,CUSTOMERS,PERCENT
0,0,1782,68.54
1,1,818,31.46


,Profile Missing,Activity Missing,Churn Missing
AGE,78,0,0
APP_SESSIONS,0,0,0
AVG_BASKET_VALUE,0,0,0
CHURNED,0,0,0
CITY,0,0,0
COMPLAINT_FLAG,0,0,0
COUPONS_REDEEMED,0,0,0
CUSTOMER_ID,0,0,0
DISTINCT_CATEGORIES,0,0,0
EMAILS_OPENED,0,0,0


,MEMBERSHIP_TIER,CUSTOMERS,CHURN_RATE
0,Gold,824,30.10
1,Platinum,302,31.13
2,Silver,1474,32.29


,CITY,CUSTOMERS,CHURN_RATE
0,Austin,266,37.59
2,Columbus,268,36.19
5,Nashville,237,33.33
1,Boise,247,31.98
9,Tulsa,295,29.83
7,Raleigh,282,29.79
4,Madison,256,29.69
8,Spokane,252,28.97
3,Denver,257,28.79
6,Portland,240,28.33


,CUSTOMER_ID,MONTH,TRANSACTIONS,TOTAL_SPEND,AVG_BASKET_VALUE,DISTINCT_CATEGORIES,PROMO_TXN_PCT,APP_SESSIONS,EMAILS_OPENED,COUPONS_REDEEMED,SUPPORT_TICKETS,COMPLAINT_FLAG
14397,501780,2024-05-01,3,116.18,38.73,2,0.07,6,1,0,0,0
16366,501780,2024-05-01,3,-116.18,38.73,2,0.07,6,1,0,0,0
15414,502065,2023-04-01,3,-70.32,23.44,2,0.22,0,2,1,0,0
26512,502065,2023-04-01,3,70.32,23.44,2,0.22,0,2,1,0,0


,CUSTOMER_ID,MONTH,TRANSACTIONS,TOTAL_SPEND,AVG_BASKET_VALUE,DISTINCT_CATEGORIES,PROMO_TXN_PCT,APP_SESSIONS,EMAILS_OPENED,COUPONS_REDEEMED,SUPPORT_TICKETS,COMPLAINT_FLAG
731,501082,2023-09-01,10,-226.93,22.69,4,0.09,5,0,0,1,0
772,500973,2024-02-01,1,-31.05,31.05,2,0.69,0,2,1,0,0
2764,501679,2024-03-01,7,-249.21,35.60,3,0.20,4,0,1,0,0
4477,500160,2023-01-01,2,-89.01,44.50,4,0.55,2,0,1,0,0
4645,501401,2023-05-01,4,-67.01,16.75,5,0.12,5,2,0,0,0
5343,501769,2023-03-01,1,-52.38,52.38,3,0.42,3,2,0,0,0
6118,501344,2023-12-01,7,-237.96,33.99,2,0.22,3,4,0,0,0
7027,501222,2023-04-01,5,-101.39,20.28,1,0.29,0,0,1,0,0
7571,502227,2024-02-01,5,-196.53,39.31,5,0.47,1,1,1,0,0
7962,501597,2023-08-01,3,-138.85,46.28,4,0.38,2,1,0,0,0


,MONTH,CUSTOMERS,TRANSACTIONS,TOTAL_SPEND,AVG_BASKET_VALUE,APP_SESSIONS,EMAILS_OPENED,COUPONS_REDEEMED,SUPPORT_TICKETS
0,2023-01-01,1467,5767,184034.600000,29.162699,2763,1453,821,233
1,2023-02-01,1493,5926,194039.178409,29.410958,2853,1496,827,231
2,2023-03-01,1514,5998,192036.120000,29.451103,2872,1446,769,239
3,2023-04-01,1507,6050,192565.270000,29.328820,2944,1474,806,233
4,2023-05-01,1523,6055,194400.070000,28.979796,3030,1478,822,238
5,2023-06-01,1534,6311,205348.608525,29.150052,3018,1497,839,246
6,2023-07-01,1538,6290,203216.977122,29.657380,3222,1548,894,235
7,2023-08-01,1553,6553,210740.850000,29.792157,3216,1631,878,249
8,2023-09-01,1569,6579,209103.600000,29.645194,3202,1606,882,246
9,2023-10-01,1577,6638,216617.437336,29.652238,3235,1623,942,246


In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

project = Path.cwd().parent

profile = pd.read_csv(project / "data" / "processed" / "customer_profile_clean.csv")
activity = pd.read_csv(project / "data" / "processed" / "monthly_activity_clean.csv")
churn = pd.read_csv(project / "data" / "processed" / "churn_label_clean.csv")

profile["SIGNUP_DATE"] = pd.to_datetime(profile["SIGNUP_DATE"])
activity["MONTH"] = pd.to_datetime(activity["MONTH"])
churn["OBSERVATION_END_DATE"] = pd.to_datetime(churn["OBSERVATION_END_DATE"])
churn["LAST_PURCHASE_DATE"] = pd.to_datetime(churn["LAST_PURCHASE_DATE"])

activity = activity[activity["TOTAL_SPEND"] >= 0].copy()

activity = (
    activity
    .sort_values(["CUSTOMER_ID", "MONTH", "TOTAL_SPEND"])
    .drop_duplicates(["CUSTOMER_ID", "MONTH"], keep="last")
    .reset_index(drop=True)
)

profile["AGE"] = profile["AGE"].fillna(profile["AGE"].median())
profile["GENDER"] = profile["GENDER"].fillna("Unknown")

profile = profile.drop(columns=["HOME_STORE_PRICE_TIER"])

activity["PROMO_TXN_PCT"] = activity["PROMO_TXN_PCT"].clip(0, 1)

activity["AVG_BASKET_VALUE"] = np.where(
    activity["TRANSACTIONS"] > 0,
    activity["TOTAL_SPEND"] / activity["TRANSACTIONS"],
    0
)

churn = churn[churn["INSUFFICIENT_HISTORY_FLAG"] == 0].copy()

profile.to_csv(project / "data" / "processed" / "customer_profile_model.csv", index=False)
activity.to_csv(project / "data" / "processed" / "monthly_activity_model.csv", index=False)
churn.to_csv(project / "data" / "processed" / "churn_label_model.csv", index=False)

pd.DataFrame({
    "Dataset": ["Profile", "Activity", "Churn"],
    "Rows": [len(profile), len(activity), len(churn)],
    "Columns": [profile.shape[1], activity.shape[1], churn.shape[1]]
})

,Dataset,Rows,Columns
0,Profile,2600,9
1,Activity,27991,12
2,Churn,2227,6
